# Advanced Pitch Statistics from Statcast

This notebook explores the advanced pitch-level data used to build pitcher and batter profiles.

**Pitcher Arsenal Stats:**
- Velocity, spin, movement by pitch type
- Pitch usage percentages
- Whiff%, CSW%, zone%, chase induced

**Batter Profile Stats:**
- Performance vs pitch types (fastball, breaking, offspeed)
- Performance vs velocity buckets (90-93, 94-96, 97+)
- Performance vs movement profiles (high rise, heavy sweep, heavy drop)
- Contact quality (exit velo, xwOBA)

In [1]:
import pandas as pd
import numpy as np

from mlb_data import (
    get_statcast_pitches,
    get_pitcher_arsenal,
    get_pitcher_pitch_type_stats,
    get_batter_pitch_stats,
    get_batter_pitch_type_stats,
    get_plate_appearances,
)

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

# Date range for data collection
# Start with a smaller range for exploration (1-2 months)
START_DATE = '2024-05-01'
END_DATE = '2024-05-05'

## 1. Raw Statcast Data

First, let's fetch the raw pitch-level data and explore what's available.

In [2]:
# Fetch raw Statcast pitches
# This may take a few minutes for large date ranges
pitches = get_statcast_pitches(START_DATE, END_DATE)

print(f"Total pitches: {len(pitches):,}")
print(f"Date range: {pitches['game_date'].min()} to {pitches['game_date'].max()}")
print(f"\nColumns ({len(pitches.columns)}):")
print(pitches.columns.tolist())

Fetching Statcast pitches from 2024-05-01 to 2024-05-05...
  2024-05-01 to 2024-05-05...
This is a large query, it may take a moment to complete


100%|██████████| 5/5 [00:09<00:00,  1.95s/it]


  Total pitches: 19,165
Total pitches: 19,165
Date range: 2024-05-01 00:00:00 to 2024-05-05 00:00:00

Columns (118):
['pitch_type', 'game_date', 'release_speed', 'release_pos_x', 'release_pos_z', 'player_name', 'batter', 'pitcher', 'events', 'description', 'spin_dir', 'spin_rate_deprecated', 'break_angle_deprecated', 'break_length_deprecated', 'zone', 'des', 'game_type', 'stand', 'p_throws', 'home_team', 'away_team', 'type', 'hit_location', 'bb_type', 'balls', 'strikes', 'game_year', 'pfx_x', 'pfx_z', 'plate_x', 'plate_z', 'on_3b', 'on_2b', 'on_1b', 'outs_when_up', 'inning', 'inning_topbot', 'hc_x', 'hc_y', 'tfs_deprecated', 'tfs_zulu_deprecated', 'umpire', 'sv_id', 'vx0', 'vy0', 'vz0', 'ax', 'ay', 'az', 'sz_top', 'sz_bot', 'hit_distance_sc', 'launch_speed', 'launch_angle', 'effective_speed', 'release_spin_rate', 'release_extension', 'game_pk', 'fielder_2', 'fielder_3', 'fielder_4', 'fielder_5', 'fielder_6', 'fielder_7', 'fielder_8', 'fielder_9', 'release_pos_y', 'estimated_ba_using_sp

In [3]:
# Key pitch characteristic columns
pitch_cols = [
    'pitch_type', 'release_speed', 'release_spin_rate',
    'pfx_x', 'pfx_z', 'release_extension',
    'plate_x', 'plate_z', 'zone',
    'description', 'events', 'type',
]

pitches[pitch_cols].head(20)

,pitch_type,release_speed,release_spin_rate,pfx_x,pfx_z,release_extension,plate_x,plate_z,zone,description,events,type
0,FF,95.5,2470,-0.82,1.23,6.8,-0.35354,2.709695,4,hit_into_play,field_out,X
1,FF,96.6,2375,-0.81,1.47,7.0,0.873445,3.056491,12,called_strike,None,S
2,SL,88.1,2585,0.29,-0.11,6.8,0.603729,1.972011,9,hit_into_play,single,X
3,FF,97.2,2456,-0.88,1.28,6.9,0.632673,3.917028,3,ball,None,B
4,FF,96.9,2465,-0.9,1.29,7.0,0.693248,2.881546,6,foul,None,S
5,SL,88.8,2687,0.53,-0.17,7.0,0.741418,1.460667,14,ball,None,B
6,SL,88.3,2463,0.31,-0.15,6.7,0.011636,2.628903,5,foul,None,S
7,SI,95.8,2379,-1.41,0.96,7.0,-0.165987,0.726876,13,hit_into_play,field_out,X
8,SL,89.2,2663,0.35,0.03,6.8,0.444063,0.45227,14,ball,None,B
9,SL,86.2,2556,0.27,0.12,6.8,-0.500545,2.94893,1,foul,None,S


In [4]:
# Pitch type distribution
print("Pitch type distribution:")
pitch_counts = pitches['pitch_type'].value_counts()
pitch_pcts = pitches['pitch_type'].value_counts(normalize=True) * 100

pd.DataFrame({
    'count': pitch_counts,
    'pct': pitch_pcts.round(1)
}).head(15)

Pitch type distribution:


,count,pct
pitch_type,,
FF,5906,30.9
SI,3083,16.1
SL,2897,15.2
CH,1806,9.4
FC,1565,8.2
ST,1402,7.3
CU,1331,7.0
FS,677,3.5
KC,343,1.8


In [5]:
# Pitch outcome descriptions
print("Pitch descriptions (outcomes):")
pitches['description'].value_counts()

Pitch descriptions (outcomes):


description
ball                       6490
foul                       3321
hit_into_play              3309
called_strike              3266
swinging_strike            1942
blocked_ball                386
foul_tip                    197
swinging_strike_blocked     128
hit_by_pitch                 48
automatic_ball               45
foul_bunt                    28
missed_bunt                   3
automatic_strike              2
Name: count, dtype: int64

## 2. Pitcher Arsenal Profiles

Aggregate pitch-level data into pitcher profiles with:
- Fastball characteristics (velocity, spin, movement)
- Breaking ball characteristics
- Offspeed characteristics
- Overall effectiveness metrics (whiff%, CSW%, etc.)

In [6]:
# Get pitcher arsenal stats (uses cached pitches)
pitcher_arsenal = get_pitcher_arsenal(
    START_DATE, END_DATE,
    min_pitches=5,
    pitches_df=pitches
)

print(f"Pitchers with arsenal data: {len(pitcher_arsenal)}")
pitcher_arsenal.head(10)

Computing pitcher arsenal stats...
  Computed arsenal for 380 pitchers
Pitchers with arsenal data: 380


,pitcher_id,pitcher_name,p_throws,total_pitches,primary_pitch,ff_usage_pct,ff_velo_avg,ff_spin_avg,ff_vmov_avg,ff_hmov_avg,ff_whiff_pct,si_usage_pct,fc_usage_pct,sl_usage_pct,cu_usage_pct,cu_velo_avg,cu_spin_avg,cu_vmov_avg,cu_hmov_avg,cu_whiff_pct,ch_usage_pct,sv_usage_pct,fs_usage_pct,kc_usage_pct,st_usage_pct,kn_usage_pct,whiff_pct,swstr_pct,csw_pct,zone_pct,...,si_velo_avg,si_spin_avg,si_vmov_avg,si_hmov_avg,si_whiff_pct,fs_velo_avg,fs_spin_avg,fs_vmov_avg,fs_hmov_avg,fs_whiff_pct,st_velo_avg,st_spin_avg,st_vmov_avg,st_hmov_avg,st_whiff_pct,ch_velo_avg,ch_spin_avg,ch_vmov_avg,ch_hmov_avg,ch_whiff_pct,kc_velo_avg,kc_spin_avg,kc_vmov_avg,kc_hmov_avg,kc_whiff_pct,kn_velo_avg,kn_spin_avg,kn_vmov_avg,kn_hmov_avg,kn_whiff_pct
0,434378,"Verlander, Justin",R,97,FF,0.577320,94.055357,2435.642857,1.567143,-0.780714,0.250000,0.000000,0.000000,0.123711,0.206186,78.210000,2710.100000,-1.089500,0.788,0.000,0.092784,0.0,0.000000,0.0,0.000000,0.0,0.173913,0.082474,0.206186,0.484536,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,445276,"Jansen, Kenley",R,15,FC,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000,1.000000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.100000,0.066667,0.133333,0.733333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,445926,"Chavez, Jesse",R,22,FC,0.000000,NaN,NaN,NaN,NaN,NaN,0.045455,0.636364,0.000000,0.272727,NaN,NaN,NaN,NaN,NaN,0.045455,0.0,0.000000,0.0,0.000000,0.0,0.250000,0.136364,0.181818,0.363636,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,450203,"Morton, Charlie",R,98,CU,0.357143,94.402857,2186.400000,0.879143,-1.121714,0.136364,0.071429,0.071429,0.000000,0.459184,81.653333,3045.444444,-0.776222,1.028,0.375,0.040816,0.0,0.000000,0.0,0.000000,0.0,0.204545,0.091837,0.326531,0.561224,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,455119,"Martin, Chris",R,10,FF,0.700000,NaN,NaN,NaN,NaN,NaN,0.000000,0.200000,0.000000,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.0,0.100000,0.0,0.000000,0.0,0.333333,0.200000,0.300000,0.300000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,458681,"Lynn, Lance",R,94,FF,0.329787,92.219355,2541.677419,0.940968,-0.226452,0.250000,0.138298,0.276596,0.000000,0.031915,NaN,NaN,NaN,NaN,NaN,0.106383,0.0,0.000000,0.0,0.117021,0.0,0.256410,0.106383,0.255319,0.457447,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,471911,"Carrasco, Carlos",R,71,SL,0.154930,NaN,NaN,NaN,NaN,NaN,0.253521,0.000000,0.309859,0.183099,NaN,NaN,NaN,NaN,NaN,0.098592,0.0,0.000000,0.0,0.000000,0.0,0.263158,0.140845,0.309859,0.436620,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,472610,"García, Luis",R,15,SI,0.000000,NaN,NaN,NaN,NaN,NaN,0.533333,0.000000,0.266667,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.0,0.200000,0.0,0.000000,0.0,0.444444,0.266667,0.333333,0.333333,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,489446,"Yates, Kirby",R,27,FF,0.629630,NaN,NaN,NaN,NaN,NaN,0.000000,0.000000,0.037037,0.000000,NaN,NaN,NaN,NaN,NaN,0.000000,0.0,0.333333,0.0,0.000000,0.0,0.181818,0.074074,0.185185,0.444444,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,493603,"Ottavino, Adam",R,53,SI,0.037736,NaN,NaN,NaN,NaN,NaN,0.396226,0.037736,0.056604,0.000000,NaN,NaN,NaN,NaN,NaN,0.132075,0.0,0.000000,0.0,0.339623,0.0,0.150000,0.056604,0.283019,0.433962,...,92.628571,2263.428571,0.541429,-1.483333,0.125,NaN,NaN,NaN,NaN,NaN,NaN,

In [7]:
# All available arsenal columns
print("Pitcher arsenal columns:")
print(pitcher_arsenal.columns.tolist())

Pitcher arsenal columns:
['pitcher_id', 'pitcher_name', 'p_throws', 'total_pitches', 'primary_pitch', 'ff_usage_pct', 'ff_velo_avg', 'ff_spin_avg', 'ff_vmov_avg', 'ff_hmov_avg', 'ff_whiff_pct', 'si_usage_pct', 'fc_usage_pct', 'sl_usage_pct', 'cu_usage_pct', 'cu_velo_avg', 'cu_spin_avg', 'cu_vmov_avg', 'cu_hmov_avg', 'cu_whiff_pct', 'ch_usage_pct', 'sv_usage_pct', 'fs_usage_pct', 'kc_usage_pct', 'st_usage_pct', 'kn_usage_pct', 'whiff_pct', 'swstr_pct', 'csw_pct', 'zone_pct', 'chase_pct_induced', 'contact_pct', 'f_strike_pct', 'fc_velo_avg', 'fc_spin_avg', 'fc_vmov_avg', 'fc_hmov_avg', 'fc_whiff_pct', 'sl_velo_avg', 'sl_spin_avg', 'sl_vmov_avg', 'sl_hmov_avg', 'sl_whiff_pct', 'si_velo_avg', 'si_spin_avg', 'si_vmov_avg', 'si_hmov_avg', 'si_whiff_pct', 'fs_velo_avg', 'fs_spin_avg', 'fs_vmov_avg', 'fs_hmov_avg', 'fs_whiff_pct', 'st_velo_avg', 'st_spin_avg', 'st_vmov_avg', 'st_hmov_avg', 'st_whiff_pct', 'ch_velo_avg', 'ch_spin_avg', 'ch_vmov_avg', 'ch_hmov_avg', 'ch_whiff_pct', 'kc_velo_avg'

In [10]:
# Summary statistics for key arsenal metrics
arsenal_stats = [
    'fb_velo_avg', 'fb_velo_max', 'fb_spin_avg', 'fb_vmov_avg',
    'brk_velo_avg', 'brk_hmov_avg', 'brk_vmov_avg',
    'off_velo_avg', 'off_velo_diff',
    'fb_usage_pct', 'brk_usage_pct', 'off_usage_pct',
    'whiff_pct', 'csw_pct', 'zone_pct', 'chase_pct_induced',
]

available_stats = [c for c in arsenal_stats if c in pitcher_arsenal.columns]
pitcher_arsenal[available_stats].describe().round(2)

,whiff_pct,csw_pct,zone_pct,chase_pct_induced
count,380.00,380.00,380.00,380.00
mean,0.24,0.28,0.50,0.28
std,0.12,0.08,0.09,0.14
min,0.00,0.00,0.28,0.00
25%,0.15,0.23,0.45,0.20
50%,0.22,0.28,0.50,0.27
75%,0.31,0.32,0.55,0.35
max,0.62,0.57,0.80,1.00


## 3. Pitcher Stats by Pitch Type

Detailed breakdown for each pitch type a pitcher throws.

In [12]:
# Get per-pitch-type stats
pitcher_pitch_types = get_pitcher_pitch_type_stats(
    START_DATE, END_DATE,
    min_pitches=5,
    pitches_df=pitches
)

print(f"Pitcher-pitch type combinations: {len(pitcher_pitch_types)}")
pitcher_pitch_types.head(15)

Computing pitcher pitch-type stats...
  Computed stats for 1109 pitcher-pitch combinations
Pitcher-pitch type combinations: 1109


,pitcher_id,pitcher_name,p_throws,pitch_type,pitch_count,usage_pct,velo_avg,velo_max,spin_avg,vmov_avg,hmov_avg,extension,whiff_pct,csw_pct,swing_pct,avg_exit_velo,avg_launch_angle,xwoba_against,xba_against
0,434378,"Verlander, Justin",R,CH,9,0.092784,84.933333,86.1,1863.666667,0.696667,-1.312222,6.033333,0.000000,0.222222,0.222222,NaN,NaN,NaN,NaN
1,434378,"Verlander, Justin",R,CU,20,0.206186,78.210000,80.2,2710.100000,-1.089500,0.788000,5.940000,0.000000,0.150000,0.350000,NaN,NaN,NaN,NaN
2,434378,"Verlander, Justin",R,FF,56,0.577320,94.055357,97.0,2435.642857,1.567143,-0.780714,5.967857,0.250000,0.250000,0.571429,88.98,31.0,0.224812,0.201427
3,434378,"Verlander, Justin",R,SL,12,0.123711,86.650000,88.4,2523.083333,0.370000,0.471667,5.975000,0.000000,0.083333,0.416667,NaN,NaN,NaN,NaN
4,445276,"Jansen, Kenley",R,FC,15,1.000000,91.813333,92.9,2600.666667,1.258667,0.622667,6.846667,0.100000,0.133333,0.666667,NaN,NaN,NaN,NaN
5,445926,"Chavez, Jesse",R,CU,6,0.272727,74.650000,75.5,2479.333333,-0.648333,1.185000,6.333333,0.000000,0.000000,0.666667,NaN,NaN,NaN,NaN
6,445926,"Chavez, Jesse",R,FC,14,0.636364,88.392857,89.9,2193.000000,1.072143,-0.191429,6.285714,0.428571,0.285714,0.500000,NaN,NaN,NaN,NaN
7,450203,"Morton, Charlie",R,CU,45,0.459184,81.653333,83.6,3045.444444,-0.776222,1.028000,6.160000,0.375000,0.400000,0.355556,NaN,NaN,NaN,NaN
8,450203,"Morton, Charlie",R,FC,7,0.071429,89.242857,90.5,2469.428571,0.480000,-0.018571,6.228571,0.000000,0.285714,0.428571,NaN,NaN,NaN,NaN
9,450203,"Morton, Charlie",R,FF,35,0.357143,94.402857,96.6,2186.400000,0.879143,-1.121714,6.214286,0.136364,0.200000,0.628571,NaN,NaN,NaN,NaN


In [13]:
# Best sliders by whiff rate
sliders = pitcher_pitch_types[pitcher_pitch_types['pitch_type'] == 'SL']
print("Top 10 sliders by whiff rate:")
sliders.nlargest(10, 'whiff_pct')[[
    'pitcher_name', 'pitch_count', 'usage_pct',
    'velo_avg', 'hmov_avg', 'vmov_avg', 'whiff_pct'
]]

Top 10 sliders by whiff rate:


,pitcher_name,pitch_count,usage_pct,velo_avg,hmov_avg,vmov_avg,whiff_pct
568,"Reid-Foley, Sean",6,0.166667,86.350000,0.473333,0.030000,1.000000
641,"McKenzie, Triston",5,0.055556,85.840000,0.390000,0.876000,1.000000
792,"Miller, Erik",10,0.192308,81.030000,-1.069000,-0.130000,1.000000
921,"Sears, JP",8,0.084211,78.737500,-0.690000,0.651250,1.000000
865,"Morejon, Adrian",9,0.450000,86.533333,-0.573333,-0.423333,0.833333
53,"Pressly, Ryan",7,0.179487,89.000000,0.478571,0.361429,0.800000
85,"Hudson, Daniel",11,0.407407,88.381818,-0.019091,0.441818,0.800000
552,"Hoffman, Jeff",14,0.411765,87.442857,0.387143,-0.170000,0.800000
758,"Doval, Camilo",8,0.571429,88.487500,0.248750,-0.072500,0.800000
911,"Walker, Ryan",15,0.535714,83.040000,1.500667,0.110667,0.714286


In [14]:
# Best changeups by whiff rate
changeups = pitcher_pitch_types[pitcher_pitch_types['pitch_type'] == 'CH']
print("Top 10 changeups by whiff rate:")
changeups.nlargest(10, 'whiff_pct')[[
    'pitcher_name', 'pitch_count', 'usage_pct',
    'velo_avg', 'vmov_avg', 'whiff_pct'
]]

Top 10 changeups by whiff rate:


,pitcher_name,pitch_count,usage_pct,velo_avg,vmov_avg,whiff_pct
259,"López, Jorge",6,0.111111,87.366667,0.215000,1.000000
263,"Musgrove, Joe",5,0.057471,87.680000,0.174000,1.000000
799,"Burnes, Corbin",8,0.084211,89.387500,0.420000,1.000000
974,"Vodnik, Victor",5,0.121951,89.920000,-0.046000,1.000000
1088,"Elder, Bryce",5,0.070423,84.780000,0.366000,1.000000
308,"Martinez, Nick",6,0.260870,80.100000,0.451667,0.800000
526,"Beeks, Jalen",9,0.409091,89.688889,0.883333,0.750000
276,"Ross, Joe",11,0.127907,90.218182,0.560909,0.666667
359,"Eflin, Zach",6,0.068182,86.500000,0.186667,0.666667
495,"Santana, Dennis",7,0.205882,88.814286,0.767143,0.666667


## 4. Batter Pitch Performance Profiles

How batters perform against different pitch characteristics:
- By pitch type group (fastball, breaking, offspeed)
- By velocity bucket (90-93, 94-96, 97+)
- By movement type (high rise, heavy sweep, heavy drop)
- By pitcher handedness (vs LHP, vs RHP)

In [18]:
# Get batter pitch profiles
batter_profiles = get_batter_pitch_stats(
    START_DATE, END_DATE,
    min_pitches=5,
    pitches_df=pitches
)

print(f"Batters with pitch profiles: {len(batter_profiles)}")
batter_profiles.head(10)

Computing batter pitch stats...
  Computed pitch stats for 388 batters
Batters with pitch profiles: 388


,batter_id,batter_name,stand,total_pitches,swing_pct,whiff_pct,contact_pct,csw_pct,zone_swing_pct,chase_pct,vs_ff_whiff_pct,vs_ff_contact_pct,vs_RHP_pitches,vs_RHP_whiff_pct,vs_RHP_xwoba,vs_si_whiff_pct,vs_si_contact_pct,vs_si_xwoba,vs_ff_xwoba,vs_sl_whiff_pct,vs_sl_contact_pct,vs_ch_whiff_pct,vs_ch_contact_pct,vs_ch_xwoba,vs_LHP_pitches,vs_LHP_whiff_pct,vs_LHP_xwoba,velo_90_93_pitches,velo_90_93_whiff_pct,velo_90_93_swing_pct,vs_st_whiff_pct,vs_st_contact_pct,velo_94_96_pitches,velo_94_96_whiff_pct,velo_94_96_swing_pct,avg_exit_velo,max_exit_velo,avg_launch_angle,xwoba,xba,barrel_pct,vs_sl_xwoba
0,453568,,L,33,0.666667,0.272727,0.727273,0.272727,0.789474,0.500000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,455117,,R,32,0.531250,0.411765,0.588235,0.375000,0.722222,0.285714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,457705,,R,37,0.432432,0.250000,0.750000,0.297297,0.684211,0.166667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,457759,,R,67,0.507463,0.117647,0.882353,0.253731,0.666667,0.352941,0.214286,0.785714,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,467793,,L,74,0.445946,0.090909,0.909091,0.216216,0.647059,0.275000,NaN,NaN,67.0,0.096774,0.295833,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,500743,,R,31,0.483871,0.066667,0.933333,0.161290,0.714286,0.294118,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,501303,,L,50,0.460000,0.434783,0.565217,0.420000,0.608696,0.333333,0.384615,0.615385,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,502054,,R,75,0.440000,0.181818,0.818182,0.186667,0.702703,0.184211,NaN,NaN,58.0,0.120000,0.334063,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,502110,,R,61,0.573770,0.371429,0.628571,0.295082,0.777778,0.411765,0.181818,0.818182,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,502671,,R,84,0.380952,0.375000,0.625000,0.345238,0.487179,0.288889,0.428571,0.571429,66.0,0.333333,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [19]:
# All available batter profile columns
print("Batter profile columns:")
print(batter_profiles.columns.tolist())

Batter profile columns:
['batter_id', 'batter_name', 'stand', 'total_pitches', 'swing_pct', 'whiff_pct', 'contact_pct', 'csw_pct', 'zone_swing_pct', 'chase_pct', 'vs_ff_whiff_pct', 'vs_ff_contact_pct', 'vs_RHP_pitches', 'vs_RHP_whiff_pct', 'vs_RHP_xwoba', 'vs_si_whiff_pct', 'vs_si_contact_pct', 'vs_si_xwoba', 'vs_ff_xwoba', 'vs_sl_whiff_pct', 'vs_sl_contact_pct', 'vs_ch_whiff_pct', 'vs_ch_contact_pct', 'vs_ch_xwoba', 'vs_LHP_pitches', 'vs_LHP_whiff_pct', 'vs_LHP_xwoba', 'velo_90_93_pitches', 'velo_90_93_whiff_pct', 'velo_90_93_swing_pct', 'vs_st_whiff_pct', 'vs_st_contact_pct', 'velo_94_96_pitches', 'velo_94_96_whiff_pct', 'velo_94_96_swing_pct', 'avg_exit_velo', 'max_exit_velo', 'avg_launch_angle', 'xwoba', 'xba', 'barrel_pct', 'vs_sl_xwoba']


In [20]:
# Summary of key batter metrics
batter_stats = [
    'whiff_pct', 'contact_pct', 'chase_pct', 'zone_swing_pct',
    'fastball_whiff_pct', 'breaking_whiff_pct', 'offspeed_whiff_pct',
    'velo_97_plus_whiff_pct', 'high_rise_whiff_pct', 'heavy_sweep_whiff_pct',
    'avg_exit_velo', 'xwoba',
]

available_stats = [c for c in batter_stats if c in batter_profiles.columns]
batter_profiles[available_stats].describe().round(3)

,whiff_pct,contact_pct,chase_pct,zone_swing_pct,avg_exit_velo,xwoba
count,388.000,388.000,388.000,388.000,2.000,2.000
mean,0.237,0.763,0.277,0.655,86.410,0.354
std,0.127,0.127,0.125,0.130,0.774,0.056
min,0.000,0.250,0.000,0.000,85.862,0.315
25%,0.155,0.700,0.200,0.576,86.136,0.335
50%,0.222,0.778,0.267,0.658,86.410,0.354
75%,0.300,0.845,0.346,0.742,86.683,0.374
max,0.750,1.000,1.000,1.000,86.957,0.394


In [21]:
# Best contact rates (lowest whiff)
print("Top 10 contact rates (lowest whiff%):")
batter_profiles.nsmallest(10, 'whiff_pct')[[
    'batter_id', 'stand', 'total_pitches',
    'whiff_pct', 'contact_pct', 'chase_pct', 'avg_exit_velo'
]]

Top 10 contact rates (lowest whiff%):


,batter_id,stand,total_pitches,whiff_pct,contact_pct,chase_pct,avg_exit_velo
37,571657,R,13,0.0,1.0,0.200000,NaN
96,621028,R,14,0.0,1.0,0.200000,NaN
100,621439,R,5,0.0,1.0,0.250000,NaN
126,641584,L,22,0.0,1.0,0.181818,NaN
139,642731,R,72,0.0,1.0,0.178571,NaN
147,644433,R,14,0.0,1.0,0.000000,NaN
195,663611,R,14,0.0,1.0,0.375000,NaN
263,668800,R,14,0.0,1.0,0.285714,NaN
274,669134,R,28,0.0,1.0,0.181818,NaN
305,672640,R,21,0.0,1.0,0.200000,NaN


In [22]:
# Batters who struggle vs 97+ velocity
if 'velo_97_plus_whiff_pct' in batter_profiles.columns:
    has_velo_data = batter_profiles['velo_97_plus_whiff_pct'].notna()
    print("Batters who struggle most vs 97+ mph:")
    batter_profiles[has_velo_data].nlargest(10, 'velo_97_plus_whiff_pct')[[
        'batter_id', 'stand', 'velo_97_plus_pitches',
        'velo_97_plus_whiff_pct', 'whiff_pct'
    ]]

In [23]:
# Batters who struggle vs breaking balls
if 'breaking_whiff_pct' in batter_profiles.columns:
    has_brk_data = batter_profiles['breaking_whiff_pct'].notna()
    print("Batters who struggle most vs breaking balls:")
    batter_profiles[has_brk_data].nlargest(10, 'breaking_whiff_pct')[[
        'batter_id', 'stand', 'breaking_pitches',
        'breaking_whiff_pct', 'whiff_pct'
    ]]

## 5. Batter Stats by Pitch Type

Detailed breakdown for each pitch type a batter faces.

In [24]:
# Get per-pitch-type stats for batters
batter_pitch_types = get_batter_pitch_type_stats(
    START_DATE, END_DATE,
    min_pitches=30,
    pitches_df=pitches
)

print(f"Batter-pitch type combinations: {len(batter_pitch_types)}")
batter_pitch_types.head(15)

Computing batter pitch-type stats...
  Computed stats for 26 batter-pitch combinations
Batter-pitch type combinations: 26


,batter_id,batter_name,stand,pitch_type,pitch_count,pct_of_pitches_seen,swing_pct,whiff_pct,contact_pct,avg_exit_velo,avg_launch_angle,xwoba,xba
0,575929,,R,FF,30,0.340909,0.466667,0.357143,0.642857,NaN,NaN,NaN,NaN
1,605141,,R,FF,34,0.465753,0.411765,0.000000,1.000000,91.437500,37.000000,0.287926,0.241668
2,606192,,R,FF,37,0.445783,0.540541,0.300000,0.700000,NaN,NaN,NaN,NaN
3,623993,,L,FF,37,0.506849,0.540541,0.100000,0.900000,87.666667,24.333333,0.385974,0.294115
4,630105,,L,FF,39,0.469880,0.589744,0.173913,0.826087,95.657143,17.428571,0.510973,0.303153
5,641857,,L,FF,47,0.489583,0.382979,0.000000,1.000000,NaN,NaN,NaN,NaN
6,642731,,R,SI,33,0.458333,0.454545,0.000000,1.000000,90.622222,5.222222,0.300074,0.249901
7,643565,,L,FF,37,0.415730,0.459459,0.176471,0.823529,NaN,NaN,NaN,NaN
8,646240,,L,FF,45,0.517241,0.400000,0.333333,0.666667,NaN,NaN,NaN,NaN
9,663368,,L,FF,30,0.500000,0.366667,0.363636,0.636364,NaN,NaN,NaN,NaN


In [25]:
# Best fastball hitters (lowest whiff on FF)
ff_stats = batter_pitch_types[batter_pitch_types['pitch_type'] == 'FF']
if len(ff_stats) > 0 and 'xwoba' in ff_stats.columns:
    ff_qualified = ff_stats[ff_stats['pitch_count'] >= 100]
    print("Best fastball hitters by xwOBA:")
    ff_qualified.nlargest(10, 'xwoba')[[
        'batter_id', 'pitch_count', 'whiff_pct', 'contact_pct',
        'avg_exit_velo', 'xwoba'
    ]]

Best fastball hitters by xwOBA:


## 6. Plate Appearance Outcomes

Extract completed plate appearances with outcomes (K, BB, 1B, 2B, 3B, HR, OUT).

In [26]:
# Get plate appearances
plate_apps = get_plate_appearances(
    START_DATE, END_DATE,
    pitches_df=pitches
)

print(f"Total plate appearances: {len(plate_apps):,}")
plate_apps.head(15)

Extracting 4,895 plate appearances...
Total plate appearances: 4,895


,game_pk,game_date,at_bat_number,inning,outs_when_up,pitcher_id,pitcher_name,p_throws,batter_id,stand,home_team,away_team,events,outcome,launch_speed,launch_angle,estimated_ba_using_speedangle,estimated_woba_using_speedangle,is_home_batter
0,747206,2024-05-05,77,9,2,656464,"Ginkel, Kevin",R,630105,L,AZ,SD,field_out,OUT,93.0,35,0.149,0.288,True
2,747206,2024-05-05,76,9,2,656464,"Ginkel, Kevin",R,665487,R,AZ,SD,single,1B,102.8,8,0.668,0.61,True
7,747206,2024-05-05,75,9,1,656464,"Ginkel, Kevin",R,650333,L,AZ,SD,field_out,OUT,72.4,9,0.263,0.238,True
13,747206,2024-05-05,74,9,0,656464,"Ginkel, Kevin",R,543309,R,AZ,SD,strikeout,K,<NA>,<NA>,<NA>,0.0,True
17,747206,2024-05-05,73,8,2,593974,"Peralta, Wandy",L,606466,R,AZ,SD,field_out,OUT,85.2,-39,0.055,0.072,True
18,747206,2024-05-05,72,8,1,593974,"Peralta, Wandy",L,545341,R,AZ,SD,strikeout,K,<NA>,<NA>,<NA>,0.0,True
23,747206,2024-05-05,71,8,0,593974,"Peralta, Wandy",L,571466,L,AZ,SD,field_out,OUT,81.3,-12,0.076,0.052,True
26,747206,2024-05-05,70,8,2,657044,"Thompson, Ryan",R,673490,R,AZ,SD,field_out,OUT,87.3,28,0.047,0.052,True
31,747206,2024-05-05,69,8,2,657044,"Thompson, Ryan",R,701538,L,AZ,SD,single,1B,102.4,-3,0.403,0.4,True
37,747206,2024-05-05,68,8,1,657044,"Thompson, Ryan",R,593428,R,AZ,SD,strikeout,K,<NA>,<NA>,<NA>,0.0,True


In [27]:
# Outcome distribution
print("PA outcome distribution:")
outcome_counts = plate_apps['outcome'].value_counts()
outcome_pcts = plate_apps['outcome'].value_counts(normalize=True) * 100

pd.DataFrame({
    'count': outcome_counts,
    'pct': outcome_pcts.round(1)
})

PA outcome distribution:


,count,pct
outcome,,
OUT,2301,47.0
K,1115,22.8
1B,663,13.5
BB,406,8.3
2B,199,4.1
HR,120,2.5
HBP,48,1.0
3B,25,0.5
OTHER,18,0.4


In [28]:
# Unique matchup counts
print(f"Unique pitchers: {plate_apps['pitcher_id'].nunique():,}")
print(f"Unique batters: {plate_apps['batter_id'].nunique():,}")

# Pitcher-batter combinations
matchups = plate_apps.groupby(['pitcher_id', 'batter_id']).size()
print(f"Unique pitcher-batter matchups: {len(matchups):,}")
print(f"\nPAs per matchup distribution:")
matchups.describe()

Unique pitchers: 382
Unique batters: 394
Unique pitcher-batter matchups: 2,976

PAs per matchup distribution:


count    2976.000000
mean        1.644825
std         0.804521
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         4.000000
dtype: float64

## 7. Save Data

Save the computed profiles for use in model training.

In [29]:
from pathlib import Path

# Create output directory
output_dir = Path('../data/statcast')
output_dir.mkdir(parents=True, exist_ok=True)

# Save profiles
pitcher_arsenal.to_csv(output_dir / 'pitcher_arsenal.csv', index=False)
pitcher_pitch_types.to_csv(output_dir / 'pitcher_pitch_types.csv', index=False)
batter_profiles.to_csv(output_dir / 'batter_profiles.csv', index=False)
batter_pitch_types.to_csv(output_dir / 'batter_pitch_types.csv', index=False)
plate_apps.to_csv(output_dir / 'plate_appearances.csv', index=False)

print(f"Saved data to {output_dir.resolve()}:")
print(f"  - pitcher_arsenal.csv: {len(pitcher_arsenal)} rows")
print(f"  - pitcher_pitch_types.csv: {len(pitcher_pitch_types)} rows")
print(f"  - batter_profiles.csv: {len(batter_profiles)} rows")
print(f"  - batter_pitch_types.csv: {len(batter_pitch_types)} rows")
print(f"  - plate_appearances.csv: {len(plate_apps)} rows")

Saved data to /Users/matthewgillies/PitcherGamePreds/data/statcast:
  - pitcher_arsenal.csv: 380 rows
  - pitcher_pitch_types.csv: 1109 rows
  - batter_profiles.csv: 388 rows
  - batter_pitch_types.csv: 26 rows
  - plate_appearances.csv: 4895 rows
